# Incremental CPM Scheduling

This notebook describes the incremental scheduling path planned for Wave 3
(issue #8). When only a few tasks change, the engine will traverse only the
affected subgraph instead of re-running the full forward/backward pass.

> **Status**: the `changed_task_ids` parameter is **not yet implemented**.
> Cells below demonstrate timing and equivalence concepts using the current
> full-recompute API. Code snippets marked **(Wave 3 API preview)** show the
> planned interface.

**Key properties (planned)**:
- Results identical to a full recompute
- Fast for small change sets on large projects
- Automatic fallback when `changed_task_ids=None` or change set > 25% of tasks

**Install**
```bash
pip install trueppm-scheduler
```

In [ ]:
import time
from datetime import date, timedelta
from trueppm_scheduler import (
    Calendar, Dependency, Project, Task, schedule,
)

## 1. Build a medium-sized FS chain project

500 tasks in a single dependency chain is a worst-case scenario for the
forward pass (no parallelism). We use this to measure full-recompute timing.

In [ ]:
def make_fs_chain(n: int, start: date = date(2026, 1, 5)) -> Project:
    """Build an n-task FS chain — worst-case for forward-pass traversal."""
    tasks = [
        Task(id=f"t{i}", name=f"Task {i}", duration=timedelta(days=1))
        for i in range(n)
    ]
    deps = [
        Dependency(predecessor_id=f"t{i}", successor_id=f"t{i+1}")
        for i in range(n - 1)
    ]
    return Project(
        id="bench",
        name=f"{n}-task chain",
        start_date=start,
        tasks=tasks,
        dependencies=deps,
        calendar=Calendar(),
    )

project = make_fs_chain(500)
print(f"Project: {len(project.tasks)} tasks, {len(project.dependencies)} dependencies")

## 2. Full recompute timing

Baseline: how long does a full recompute take on this 500-task chain?  
Wave 3 target: full recompute < 2 000 ms; incremental (5-task change) < 200 ms.

In [ ]:
# Warm up (import-time costs)
schedule(project)

# Full recompute
t0 = time.perf_counter()
result_full = schedule(project)
full_ms = (time.perf_counter() - t0) * 1000

print(f"Full recompute: {full_ms:.1f} ms  (target: < 2000 ms)")

### Wave 3 API preview — incremental call

Once `changed_task_ids` is implemented, the timing comparison will look like:

```python
# Wave 3 only — not yet available
changed = [f"t{i}" for i in range(100, 105)]
result_incr = schedule(project, changed_task_ids=changed)

print(f"Full recompute:          {full_ms:.1f} ms")
print(f"Incremental (5 changed): {incr_ms:.1f} ms")
print(f"Speedup: {full_ms / incr_ms:.1f}x")
```

## 3. Result consistency (full recompute)

Running full recompute twice must produce identical results. The incremental
equivalence fuzz test (`tests/test_incremental_equivalence.py`) in Wave 3 will
additionally assert this across 1 000 random change scenarios.

In [ ]:
result_full2 = schedule(project)

mismatches = []
for tf, tf2 in zip(result_full.tasks, result_full2.tasks):
    if tf.early_start != tf2.early_start or tf.early_finish != tf2.early_finish:
        mismatches.append(tf.id)

if mismatches:
    print(f"MISMATCH on tasks: {mismatches}")
else:
    print("Full recompute is deterministic: both runs agree on all task dates ✓")

assert result_full.project_finish == result_full2.project_finish
print(f"Project finish: {result_full.project_finish}")

## 4. Planned fallback behaviour (Wave 3)

Two conditions will trigger an automatic fallback to full recompute:

1. `changed_task_ids=None` — explicit full recompute (same as today's API)
2. `len(changed_task_ids) / len(project.tasks) > 0.25` — change set > 25%;
   subgraph extraction overhead exceeds the savings

```python
# Wave 3 only — not yet available

# Case 1: None → full recompute
result_none = schedule(project, changed_task_ids=None)
assert result_none.project_finish == result_full.project_finish

# Case 2: > 25% changed → automatic full recompute
many_changed = [f"t{i}" for i in range(0, 200)]  # 200/500 = 40%
result_many = schedule(project, changed_task_ids=many_changed)
assert result_many.project_finish == result_full.project_finish
```

## 5. Using incremental scheduling in practice (Wave 3)

The typical call site will be inside a Celery task that handles
`PATCH /tasks/<id>/` requests:

```python
# packages/api/src/trueppm_api/apps/scheduling/tasks.py (Wave 3)
from trueppm_scheduler import schedule

@app.task
def recalculate_schedule(
    project_id: str,
    changed_task_ids: list[str] | None = None,
):
    project = build_scheduler_project(project_id)  # serialise from DB
    result  = schedule(project, changed_task_ids=changed_task_ids)
    apply_result_to_db(project_id, result)          # write ES/EF back
    broadcast_schedule_updated(project_id)          # WS event
```

The view layer passes `changed_task_ids` when the PATCH touches `duration`,
`early_start`, or dependencies — the three fields that affect the CPM graph.

## 6. Performance targets and CI guard

| Scenario | Wave 3 target |
|----------|---------------|
| 500-task chain, full recompute | < 2 000 ms |
| 500-task chain, 5-task incremental change | < 200 ms |

The `scheduler:bench` CI job in `.gitlab-ci.yml` enforces the full-recompute limit
today. The incremental bench will be added in issue #8.

```bash
# Run locally
cd packages/scheduler
pytest tests/test_bench.py -v
```

In [ ]:
# Quick local bench: full recompute only (incremental bench added in Wave 3)
import statistics

RUNS = 10
full_times = []

for _ in range(RUNS):
    t0 = time.perf_counter()
    schedule(project)
    full_times.append((time.perf_counter() - t0) * 1000)

p95_idx = max(0, int(RUNS * 0.95) - 1)
print(
    f"Full recompute — median {statistics.median(full_times):.1f} ms, "
    f"p95 {sorted(full_times)[p95_idx]:.1f} ms  (target: < 2000 ms)"
)